# Designing H3 HA constructs
Here, I will design HA genes to be ordered from Twist that contain regions that overlap with 
* 2851 cut vector
* fixed c-terminal endo domain, containing H3 and WSN recoded sequence, and a 16-nucleotide barcode
I will then save the expected full plasmid sequence as a GenBank file. 

Our H3 ectodomain sequences must have overlapping regions with WSN signal peptide and packaging signal, such that we can assemble them into the vector 2851 with the endodomain/barcoded region that I will clone separately. 

Each ectodomain will include HA-aa-1 to HA-aa-504 with the following regions added:
* 3' WSN signal peptide: catttgtagctacagatgcagacaca
* 5' H3/WSN packaging signal: gttgagctgaagtcaggatac
We will order these from Twist as gBlocks.

***This code assumes you've aligned your sequences, trimmed extra sequence upstream of ATG, and checked for insertions/deletions/other strangeness.***

Author: Caroline Kikawa

In [2]:
# Import relevant packages
import os
import Bio
from Bio import SeqIO
import pandas as pd

# ID input and output
datadir = '../data'
resultsdir = '../results'

os.makedirs(datadir, exist_ok=True)
os.makedirs(resultsdir, exist_ok=True)

In [3]:
# ID alignment files
# Code expects these to be aligned and have the 3' NCR deleted
library_strains_file = os.path.join(datadir, 'gisaid_download/gisaid_HA_nuc_aligned.fasta')

# Initialize empty list of sequence constructs
library_H3_constructs = []

# Define signal peptide and packaging sequences to add to each construct
WSN_signal_peptide_overlap = 'catttgtagctacagatgcagacaca'
H3_WSN_packaging_overlap = 'gttgagctgaagtcaggatac'

# Iterate through H3 ectodomains
# Trim 3' packaging signal and 3' transmembrane regions
# Add WSN signal peptide and H3-WSN TM/packaging sequences
fasta_sequences = SeqIO.parse(open(library_strains_file),'fasta')
for fasta in fasta_sequences:
    name, sequence = fasta.id, str(fasta.seq)
    
    # Check that sequences start with start codon (ie., is 3' NCR trimmed?)
    if (sequence[0:3]) != 'atg':
        print(f'! unexpected initial codon in {name}')
    
    # Build construct
    HA_ectodomain = sequence[48:1560]
    construct = WSN_signal_peptide_overlap + HA_ectodomain + H3_WSN_packaging_overlap
    
    # More succinct name to put on order sheet
    shortname = (name.split('|')[0]
                 # These names are too long and need additional shortening
                 # .replace('Saint-Petersburg', 'Saint-Peter')
                )
        
    # Add construct to list
    library_H3_constructs.append([shortname, construct])

# Change names to Nextstrain names
# These names are direct from GISAID, and contain typos/incomplete information
name_dict = {
    'A/Oman/CPHL_7248310/2024/1-1701': 'A/Oman/CPHL_7248310/2024',
    'A/Shaanxi-Yuyang/1204/2024/30-1762': 'A/Shaanxi-Yuyang/1204/2024',
    'A/Lisboa/124/2024/1-1718': 'A/Lisbon/124/2024',
    'A/British/18-1737': 'A/BritishColumbia/RV00920/2023',
    'A/DistrictOfColumbia/27/2023/1-1701': 'A/DistrictOfColumbia/27/2023'
}

library_H3_constructs = [[name_dict.get(item, item) for item in sublist] for sublist in library_H3_constructs]

## Write ordersheet for currently circulating library strains
This order was placed in November 2023

In [4]:
# Make dataframe for ordersheet
library_H3_constructs_df = pd.DataFrame(library_H3_constructs, columns = ['name', 'sequence'])

# Save ordersheet to results
outfile = os.path.join(resultsdir, '2025-02-24_library_constructs_ordersheet.csv')
library_H3_constructs_df.to_csv(outfile, index=False)
print(f'saving ordersheet to {outfile}...')

# Preview df 
print('here is a preview of the ordersheet')
library_H3_constructs_df.head()

saving ordersheet to ../results/2025-02-24_library_constructs_ordersheet.csv...
here is a preview of the ordersheet


,name,sequence
0,A/Shaanxi-Yuyang/1204/2024,catttgtagctacagatgcagacacacaaaaaatacctggaaatga...
1,A/Oman/CPHL_7248310/2024,catttgtagctacagatgcagacacacaaaaaatacctggaaatga...
2,A/DistrictOfColumbia/27/2023,catttgtagctacagatgcagacacacaaaaaatacctggaaatga...
3,A/Lisbon/124/2024,catttgtagctacagatgcagacacacaaaaaatacctggaaatga...
4,A/BritishColumbia/RV00920/2023,catttgtagctacagatgcagacacacaaaaaatacctggaaatga...


## Saving expected sequence as FASTA files
I can use these files as input in Benchling for sequence confirmation

In [21]:
mapsdir = os.path.join(resultsdir, 'construct_maps')
os.makedirs(mapsdir, exist_ok=True)

# save sequence upstream and downstream (including expected overlap)
upstream_vector = 'actcttcctttttcaatattattgaagcatttatcagggttattgtctcatgagcggatacatatttgaatgtatttagaaaaataaacaaaagagtttgtagaaacgcaaaaaggccatccgtcaggatggccttctgcttaatttgatgcctggcagtttatggcgggcgtcctgcccgccaccctccgggccgttgcttcgcaacgttcaaatccgctcccggcggatttgtcctactcaggagagcgttcaccgacaaacaacagataaaacgaaaggcccagtctttcgactgagcctttcgttttatttgatgcctggcagttccctactctcgcatggggagaccccacactaccatcggcgctacggcgtttcacttctgagttcggcatggggtcaggtgggaccaccgcgctactgccgccaggcaaattctgttttatcagaccgcttctgcgttctgatttaatctgtatcaggctgaaaattttttttcatccgccaaaacagccaaggcggccgcgctagcggccgatccccaaaaaaaaaaaaaaaaaaagagtccagagtggccccgccgctccgcgccggggggggggggggggggggacactttcggacatctggtcgacctccagcatcgggggaaaaaaaaaaacaaagtgtcgcccggagtactggtcgacctccgaagttgggggggagcaaaagcaggggaaaataaaaacaaccaaaatgaaggcaaaactactggtcctgttatatgcatttgtagctacagatgcagacaca'
endodomain = 'gttgagctgaagtcaggatacaaagattggatcctatggatttcctttgccATGtcTtgCttCCtActGtgCgtAgcACtACtAggCttTatTatgtgggcGtgTcaGaaAggCtcCCtAcaAtgTCgGatTtgTatTTAATAG'
barcode = 'NNNNNNNNNNNNNNNN'
downstream_vector = 'agatcggaagagcgtcgtgtagggaaagagtgtgcggccgctatctactcaactgtcgccagttcactggtgctttaggtctccctgggggcaatcagtttctggatgtgttctaatgggtctttgcagtgcagaatatgcatctgagattaggatttcagaaatataaggaaaaacacccttgtttctactaataacccggcggcccaaaatgccgactcggagcgaaagatatacctcccccggggccgggaggtcgcgtcaccgaccacgccgccggcccaggcgacgcgcgacacggacacctgtccccaaaaacgccaccatcgcagccacacacggagcgcccggggccctctggtcaaccccaggacacacgcgggagcagcgccgggccggggacgccctcccggccgcccgtgccacacgcagggggccggcccgtgtctccagagcgggagccggaagcattttcggccggcccctcctacgaccgggacacacgagggaccgaaggccggccaggcgcgacctctcgggccgcacgcgcgctcagggagcgctctccgactccgcacggggactcgccagaaaggatcgtgatctgcattaatgaatcaggggataacgcaggaaagaacatgtgagcaaaaggccagcaaaaggccaggaaccgtaaaaaggccgcgttgctggcgtttttccataggctccgcccccctgacgagcatcacaaaaatcgacgctcaagtcagaggtggcgaaacccgacaggactataaagataccaggcgtttccccctggaagctccctcgtgcgctctcctgttccgaccctgccgcttaccggatacctgtccgcctttctcccttcgggaagcgtggcgctttctcatagctcacgctgtaggtatctcagttcggtgtaggtcgttcgctccaagctgggctgtgtgcacgaaccccccgttcagcccgaccgctgcgccttatccggtaactatcgtcttgagtccaacccggtaagacacgacttatcgccactggcagcagccactggtaacaggattagcagagcgaggtatgtaggcggtgctacagagttcttgaagtggtggcctaactacggctacactagaagaacagtatttggtatctgcgctctgctgaagccagttaccttcggaaaaagagttggtagctcttgatccggcaaacaaaccaccgctggtagcggtggtttttttgtttgcaagcagcagattacgcgcagaaaaaaaggatctcaagaagatcctttgatcttttctacggggtctgacgctcagtggaacgaaaactcacgttaagggattttggtcatgagattatcaaaaaggatcttcacctagatccttttaaattaaaaatgaagttttaaatcaatctaaagtatatatgagtaaacttggtctgacagttaccaatgcttaatcagtgaggcacctatctcagcgatctgtctatttcgttcatccatagttgcctgactccccgtcgtgtagataactacgatacgggagggcttaccatctggccccagtgctgcaatgataccgcgagacccacgctcaccggctccagatttatcagcaataaaccagccagccggaagggccgagcgcagaagtggtcctgcaactttatccgcctccatccagtctattaattgttgccgggaagctagagtaagtagttcgccagttaatagtttgcgcaacgttgttgccattgctacaggcatcgtggtgtcacgctcgtcgtttggtatggcttcattcagctccggttcccaacgatcaaggcgagttacatgatcccccatgttgtgcaaaaaagcggttagctccttcggtcctccgatcgttgtcagaagtaagttggccgcagtgttatcactcatggttatggcagcactgcataattctcttactgtcatgccatccgtaagatgcttttctgtgactggtgagtactcaaccaagtcattctgagaatagtgtatgcggcgaccgagttgctcttgcccggcgtcaacacgggataataccgcgccacatagcagaactttaaaagtgctcatcattggaaaacgttcttcggggcgaaaactctcaaggatcttaccgctgttgagatccagttcgatgtaacccactcgtgcacccaactgatcttcagcatcttttactttcaccagcgtttctgggtgagcaaaaacaggaaggcaaaatgccgcaaaaaagggaataagggcgacacggaaatgttgaatactcat'

# Initialize numbering scheme in library
libID = 80

# Initialize plasmid log numbering 
logID = 5081
lib_ID_list = []

# initialize list for protein constructs
library_H3_protein_constructs = []

# Do this for both vaccine and library files
files = [
    # vaccines_strains_file, 
    library_strains_file,
    # new_vaccine_strains_file
]

for f in files:
    fasta_sequences = SeqIO.parse(open(f),'fasta')
    for fasta in fasta_sequences:
        name, sequence = fasta.id, str(fasta.seq)

        # Check that sequences start with start codon (ie., is 3' NCR trimmed?)
        if (sequence[0:3]) != 'atg':
            print(f'! unexpected initial codon in {name}')
            
        # Build construct
        HA1 = sequence[48:1560]
        construct_seq = (upstream_vector + 
                         HA1 + 
                         endodomain + 
                         barcode + 
                         downstream_vector)
        
        # Information for library
        print(name)
        strain_name, epi = name.split('|')
        print(strain_name)
        # epi = name.split('|')[1]
        
        # Rename egg-passaged Cambodia strain so it's not the exact match to the cell-passaged strain
        # if epi == 'EPI_ISL_806547':
        #     strain_name = 'A/Cambodia/e0826360/2020_egg'

        # # Rename A/Singapore/NUH0526/2023 to A/Massachusetts/18/2022
        # # Exact amino acid match 
        # if strain_name == 'A/Singapore/NUH0526/2023':
        #     strain_name = 'A/Massachusetts/18/2022'
        
        # # Need to reindex log ID for A/Thailand/8/2022, which was ordered 
        # # after all other strains in library
        # if epi == 'EPI_ISL_16014504':
        #     logID = 4636
            
        # Make a name with underscores
        underscore_name = ((strain_name)
                     .replace('/','_')
                     .replace('(', '')
                     .replace(')', '')
                    )
        
        # Each strain has 3 barcodes
        for n in list(range(1,4)):
            
            # Write plasmid name
            plasmid_name = (str(logID) +
                            '_phh_' +
                            underscore_name.replace('_','-') + 
                            '_HA_WSNflank_bc' +
                            str(n))
            # Save to list
            lib_ID_list.append([libID, plasmid_name, strain_name, epi, construct_seq])
            # Add to plasmid log ID counter
            logID += 1
                

        # Write each map to separate output
        fasta_name = str(libID) + '_' + underscore_name + '_H3HAlibraryConstruct'
        identifier_line = ">" + strain_name + "\n"

        output_path = os.path.join(mapsdir, f'{fasta_name}.fasta')
        output_file = open(output_path,'w')
        output_file.write(identifier_line)
        sequence_line = construct_seq + "\n"
        output_file.write(sequence_line)
        print(f'saving {strain_name} full plasmid sequence to {output_path}...\n')
        output_file.close()

        # Add to counter
        libID += 1
        
        # ID construct protein sequences
        nuc_sequence = (construct_seq[741:2310+len(endodomain)])
        prot_sequence = str(Bio.Seq.Seq(nuc_sequence).translate())
        library_H3_protein_constructs.append([underscore_name, prot_sequence])

# Save construct protein sequences
outfile = os.path.join(resultsdir, '2023-2024_H3_library_constructs_protein.fasta')
print(f'saving construct protein sequences to {outfile}...')  
with open(outfile, 'w') as f:
    for item in library_H3_protein_constructs:
        name = item[0]
        seq = item[1]
        f.write('>' + name + '\n')
        f.write(seq + '\n')

A/Shaanxi-Yuyang/1204/2024/30-1762|EPI_ISL_19291297
A/Shaanxi-Yuyang/1204/2024/30-1762
saving A/Shaanxi-Yuyang/1204/2024/30-1762 full plasmid sequence to ../results/construct_maps/80_A_Shaanxi-Yuyang_1204_2024_30-1762_H3HAlibraryConstruct.fasta...

A/Oman/CPHL_7248310/2024/1-1701|EPI_ISL_19631328
A/Oman/CPHL_7248310/2024/1-1701
saving A/Oman/CPHL_7248310/2024/1-1701 full plasmid sequence to ../results/construct_maps/81_A_Oman_CPHL_7248310_2024_1-1701_H3HAlibraryConstruct.fasta...

A/DistrictOfColumbia/27/2023/1-1701|EPI_ISL_19513110
A/DistrictOfColumbia/27/2023/1-1701
saving A/DistrictOfColumbia/27/2023/1-1701 full plasmid sequence to ../results/construct_maps/82_A_DistrictOfColumbia_27_2023_1-1701_H3HAlibraryConstruct.fasta...

A/Lisboa/124/2024/1-1718|EPI_ISL_18991669
A/Lisboa/124/2024/1-1718
saving A/Lisboa/124/2024/1-1718 full plasmid sequence to ../results/construct_maps/83_A_Lisboa_124_2024_1-1718_H3HAlibraryConstruct.fasta...

A/BritishColumbia/RV00920/2023|EPI_ISL_18586451
A/Br

In [22]:
outfile = os.path.join(resultsdir, 'library_ID_sheet.csv')
library_ID_df = pd.DataFrame(lib_ID_list, columns = ['lib-id', 'plasmid', 'strain-name', 'epi', 'expected-seq'])
library_ID_df.to_csv(outfile, index=False)
print(f'saving library info sheet to {outfile}...')
library_ID_df 

saving library info sheet to ../results/library_ID_sheet.csv...


,lib-id,plasmid,strain-name,epi,expected-seq
0,80,5081_phh_A-Shaanxi-Yuyang-1204-2024-30-1762_HA...,A/Shaanxi-Yuyang/1204/2024/30-1762,EPI_ISL_19291297,actcttcctttttcaatattattgaagcatttatcagggttattgt...
1,80,5082_phh_A-Shaanxi-Yuyang-1204-2024-30-1762_HA...,A/Shaanxi-Yuyang/1204/2024/30-1762,EPI_ISL_19291297,actcttcctttttcaatattattgaagcatttatcagggttattgt...
2,80,5083_phh_A-Shaanxi-Yuyang-1204-2024-30-1762_HA...,A/Shaanxi-Yuyang/1204/2024/30-1762,EPI_ISL_19291297,actcttcctttttcaatattattgaagcatttatcagggttattgt...
3,81,5084_phh_A-Oman-CPHL-7248310-2024-1-1701_HA_WS...,A/Oman/CPHL_7248310/2024/1-1701,EPI_ISL_19631328,actcttcctttttcaatattattgaagcatttatcagggttattgt...
4,81,5085_phh_A-Oman-CPHL-7248310-2024-1-1701_HA_WS...,A/Oman/CPHL_7248310/2024/1-1701,EPI_ISL_19631328,actcttcctttttcaatattattgaagcatttatcagggttattgt...
5,81,5086_phh_A-Oman-CPHL-7248310-2024-1-1701_HA_WS...,A/Oman/CPHL_7248310/2024/1-1701,EPI_ISL_19631328,actcttcctttttcaatattattgaagcatttatcagggttattgt...
6,82,5087_phh_A-DistrictOfColumbia-27-2023-1-1701_H...,A/DistrictOfColumbia/27/2023/1-1701,EPI_ISL_19513110,actcttcctttttcaatattattgaagcatttatcagggttattgt...
7,82,5088_phh_A-DistrictOfColumbia-27-2023-1-1701_H...,A/DistrictOfColumbia/27/2023/1-1701,EPI_ISL_19513110,actcttcctttttcaatattattgaagcatttatcagggttattgt...
8,82,5089_phh_A-DistrictOfColumbia-27-2023-1-1701_H...,A/DistrictOfColumbia/27/2023/1-1701,EPI_ISL_19513110,actcttcctttttcaatattattgaagcatttatcagggttattgt...
9,83,5090_phh_A-Lisboa-124-2024-1-1718_HA_WSNflank_bc1,A/Lisboa/124/2024/1-1718,EPI_ISL_18991669,actcttcctttttcaatattattgaagcatttatcagggttattgt...
